In [3]:
import backtrader as bt

ModuleNotFoundError: No module named 'backtrader'

In [ ]:
# 将TqSdk的数据列名，映射为Backtrader可识别的列名
column_mapping = {
    'open': 'open',
    'high': 'high',
    'low': 'low',
    'close': 'close',
    'volume': 'volume',
    'open_oi': 'openinterest'  # 持仓量
}

# 筛选并重命名列
df_bt = df[list(column_mapping.keys())].rename(columns=column_mapping)

# 确保所有必要的列都存在，缺失的列用0填充
required_columns = ['open', 'high', 'low', 'close', 'volume', 'openinterest']
for col in required_columns:
    if col not in df_bt.columns:
        df_bt[col] = 0  # 如果原数据中没有持仓量，用0填充

# 再次检查数据
print(df_bt.tail())

In [ ]:
df_bt

In [ ]:
# 1. 定义一个简单的双均线金叉死叉策略
class SmaCrossStrategy(bt.Strategy):
    params = dict(
        pfast=10,   # 快速均线周期
        pslow=30    # 慢速均线周期
    )

    def __init__(self):
        # 计算快慢两条均线
        sma1 = bt.ind.SMA(period=self.p.pfast)
        sma2 = bt.ind.SMA(period=self.p.pslow)
        # 生成均线交叉信号
        self.crossover = bt.ind.CrossOver(sma1, sma2)

    def next(self):
        # 如果当前没有持仓，且出现金叉信号，则买入
        if not self.position:
            if self.crossover > 0:
                self.buy()
        # 如果持有多头，且出现死叉信号，则平仓
        elif self.crossover < 0:
            self.close()

# 2. 创建Backtrader回测引擎
cerebro = bt.Cerebro()
# 添加策略
cerebro.addstrategy(SmaCrossStrategy)

# 3. 将Pandas DataFrame转换为Backtrader数据源
data = bt.feeds.PandasData(
    dataname=df_bt,
    datetime=None,  # 因为索引已经是datetime，这里设为None
    open='open',
    high='high',
    low='low',
    close='close',
    volume='volume',
    openinterest='openinterest'
)
cerebro.adddata(data)

# 4. 设置初始资金
cerebro.broker.setcash(100000.0)

# 5. 运行回测
print('初始资金: %.2f' % cerebro.broker.getvalue())
cerebro.run()
print('最终资金: %.2f' % cerebro.broker.getvalue())

# 6. 绘制回测结果
cerebro.plot()